In [1]:
import pickle 
with open ('/scratch/ae2195/CASPIAN_MAML/data/processed/AD/val_tensor_batches_SUPPORT_K_2_QUERY_K_2_edge_case_solved.pkl', 'rb') as f:# rb reading in binary mode
    tensor_batches = pickle.load(f)



# Examine the structure 
print (f"Type: {type(tensor_batches)}")
print (f"Task IDs: {list(tensor_batches.keys())}")

# Examine all tasks and all batches 
for task_id in tensor_batches.keys(): 
    print (f"\nTask {task_id}:")
    print (f"           Number of batches: {len(tensor_batches[task_id])} ")

    # loop through each batch in this task 
    for batch_idx, batch in enumerate (tensor_batches[task_id]):
        print (f" Batch {batch_idx}  has {len(batch)} tensors: ")
        for i, tensor in enumerate (batch):         # creates a pair of (index, item) from the batch with item being the actual tensor object at the i position
            print (f"       Tensor {i}: shape {tensor.shape}, dtype {tensor.dtype}")



# one TASK is missing from '/scratch/ae2195/CASPIAN_MAML/data/processed/AD/val_tensor_batches_SUPPORT_K_2_QUERY_K_2.pkl' 


2025-09-14 17:33:42.107623: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-09-14 17:33:42.107660: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-09-14 17:33:42.114353: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-09-14 17:33:42.126206: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-09-14 17:33:43.568323: W tensorflow/compiler/tf2

Type: <class 'dict'>
Task IDs: [1, 2, 3, 4, 5]

Task 1:
           Number of batches: 10 
 Batch 0  has 5 tensors: 
       Tensor 0: shape (2, 1024, 1024, 1), dtype <dtype: 'float32'>
       Tensor 1: shape (2, 1024, 1024, 4), dtype <dtype: 'float32'>
       Tensor 2: shape (2, 1024, 1024, 1), dtype <dtype: 'float32'>
       Tensor 3: shape (2, 1024, 1024, 4), dtype <dtype: 'float32'>
       Tensor 4: shape (2, 1), dtype <dtype: 'float32'>
 Batch 1  has 5 tensors: 
       Tensor 0: shape (2, 1024, 1024, 1), dtype <dtype: 'float32'>
       Tensor 1: shape (2, 1024, 1024, 4), dtype <dtype: 'float32'>
       Tensor 2: shape (2, 1024, 1024, 1), dtype <dtype: 'float32'>
       Tensor 3: shape (2, 1024, 1024, 4), dtype <dtype: 'float32'>
       Tensor 4: shape (2, 1), dtype <dtype: 'float32'>
 Batch 2  has 5 tensors: 
       Tensor 0: shape (2, 1024, 1024, 1), dtype <dtype: 'float32'>
       Tensor 1: shape (2, 1024, 1024, 4), dtype <dtype: 'float32'>
       Tensor 2: shape (2, 1024, 1024, 1

In [ ]:
# code bellow does not work yet... 


import os, re, glob
import numpy as np
import pandas as pd
import tensorflow as tf
import keras
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


AD_EVAL_DATASET    = "/scratch/ae2195/CASPIAN_MAML/data/raw/AD_slr_0.5m/test/FSL_MAML_AD_test_ds_split_1"
SF_EVAL_DATASET    = "/scratch/ae2195/CASPIAN_MAML/data/raw/SF_slr_1m/test/FSL_MAML_SF_test_ds_split_1"

def map_eval(f,x,y,yf):
    return tf.py_function(
        clear_ds_eval,                      # Call the Python function to process one example
        inp=[f,x,y,yf],                     # Inputs: filename, input image, label, label flattened
        Tout=[                              # Output types:
            tf.string,                      # scenario name as string
            tf.float32,                     # input image tensor with shape (H, W, 1)
            tf.float32,                     # label tensor with shape (H, W, 1)
            tf.bool,                        # region mask (bool grid)
            tf.float32                      # sea level rise value (float)
        ]
    )

# Load the evaluation dataset from disk, then Apply the map function to each example using map_eval (which calls clear_ds_eval)
ad_eval_dataset = tf.data.Dataset.load(AD_EVAL_DATASET).map(map_eval)
sf_eval_dataset = tf.data.Dataset.load(SF_EVAL_DATASET).map(map_eval)

ds_eval = ad_eval_dataset.concatenate(sf_eval_dataset).batch(1) # Group the dataset into batches of size 1 (i.e., 1 scenario at a time)


def ds_count(ds):
    return int(ds.reduce(tf.constant(0, tf.int64), lambda x, _: x + 1).numpy())

n_eval = ds_count(ds_eval)
print("Number of scenarios in ds_eval:", n_eval)

